# Resume RAG — batch metadata extraction + chunking

Runs the pipeline defined in `resume_rag.py`: discover resume files under `root_dir/resumes/`, batch-extract metadata via OpenRouter, section-aware chunk each resume, and write `extractions.json` / `chunks.json`.

In [1]:
import json

import config
from resume_rag import (
    build_chunks,
    build_extraction_model,
    discover_resume_files,
    extract_fields_batch,
)

C:\Users\Harsha\sde-dev\airtribe-dev\airtribe-bel_c20\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Harsha\sde-dev\airtribe-dev\airtribe-bel_c20\rag_profile_match\config.py:17: UserWarning: LANGSMITH_TRACING is not set. Tracing will be disabled. If you want to enable tracing, please set LANGSMITH_TRACING to "true" in the environment variables or in a .env file.
  warn("LANGSMITH_TRACING is not set. Tracing will be disabled. If you want to enable tracing, please set LANGSMITH_TRACING to \"true\" in the environment variables or in a .env file.")


## 1. Discover resume files

In [2]:
entries = discover_resume_files()
print(f"Found {len(entries)} resume files")
assert any(e["name"] == "resume_john_doe.pdf" for e in entries), "expected the loose resume_john_doe.pdf to be discovered"
entries[:3]

INFO: list_files(directory='resumes', extension=None)
Found 31 resume files


[{'name': 'analytics_raj.txt',
  'path': 'resumes\\data\\analytics_raj.txt',
  'extension': '.txt',
  'size_bytes': 996,
  'modified': '2026-08-26T02:34:39.505976+00:00'},
 {'name': 'bi_leo.pdf',
  'path': 'resumes\\data\\bi_leo.pdf',
  'extension': '.pdf',
  'size_bytes': 1839,
  'modified': '2026-08-26T02:34:39.506980+00:00'},
 {'name': 'data_engineer_neha.txt',
  'path': 'resumes\\data\\data_engineer_neha.txt',
  'extension': '.txt',
  'size_bytes': 993,
  'modified': '2026-08-26T03:57:14.894468+00:00'}]

## 2. Batch-extract metadata via OpenRouter

Builds one shared, retry-wrapped extraction model, then processes `entries` in batches of `config.EXTRACTION_BATCH_SIZE` — each batch issues one concurrent `model.batch()` call (`max_concurrency=config.EXTRACTION_MAX_CONCURRENCY`).

In [3]:
extraction_model = build_extraction_model()
records = extract_fields_batch(entries, extraction_model=extraction_model)
print(f"Extracted metadata for {len(records)} resumes")

incorrect startxref pointer(1)


parsing for Object Streams


incorrect startxref pointer(1)


parsing for Object Streams


INFO: Extracting metadata for batch 1 (10 files)
INFO: read_file(filepath='resumes\\data\\analytics_raj.txt')
INFO: read_file(filepath='resumes\\data\\bi_leo.pdf')
INFO: read_file(filepath='resumes\\data\\data_engineer_neha.txt')
INFO: read_file(filepath='resumes\\data\\data_scientist_hana.docx')
INFO: read_file(filepath='resumes\\data\\ml_sara.docx')
INFO: read_file(filepath='resumes\\data\\product_analyst_owen.pdf')
INFO: read_file(filepath='resumes\\design\\brand_nina.pdf')
INFO: read_file(filepath='resumes\\design\\design_lead_priyanka.pdf')
INFO: read_file(filepath='resumes\\design\\graphic_designer_liam.docx')
INFO: read_file(filepath='resumes\\design\\product_design_zoe.txt')


...

incorrect startxref pointer(1)


parsing for Object Streams


INFO: Extracting metadata for batch 3 (10 files)
INFO: read_file(filepath='resumes\\marketing\\growth_carol.txt')
INFO: read_file(filepath='resumes\\marketing\\product_marketing_elena.docx')
INFO: read_file(filepath='resumes\\marketing\\seo_specialist_ivan.txt')
INFO: read_file(filepath='resumes\\marketing\\social_ken.pdf')
INFO: read_file(filepath='resumes\\resume_john_doe.pdf')
INFO: read_file(filepath='resumes\\sales\\account_exec_marcus.pdf')
INFO: read_file(filepath='resumes\\sales\\customer_success_grace.docx')
INFO: read_file(filepath='resumes\\sales\\enterprise_dan.txt')
INFO: read_file(filepath='resumes\\sales\\partnerships_omar.pdf')
INFO: read_file(filepath='resumes\\sales\\sales_engineer_victor.txt')


INFO: Extracting metadata for batch 4 (1 files)
INFO: read_file(filepath='resumes\\sales\\smb_lena.docx')


Extracted metadata for 31 resumes


## 3. Write extractions.json and sanity-check a sample record

In [4]:
with open("extractions.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

records[0]

{'file_path': 'resumes\\data\\analytics_raj.txt',
 'file_name': 'analytics_raj.txt',
 'dept': 'data',
 'candidate_name': 'Raj Patel',
 'skills': ['sql', 'tableau', 'python', 'ab testing', 'data modeling'],
 'total_experience_years': 7.0,
 'education_level': 'Bachelors',
 'sections': {'header': 'Raj Patel\nData Analyst\nraj.patel@example.com | Toronto, Canada\n\nData analyst with 5 years turning raw operational data into dashboards and\nrecommendations for product and operations teams.',
  'skills': 'SQL, Tableau, Python (pandas), A/B test analysis, data modeling',
  'experience': 'Senior Data Analyst, Millbrook Retail (2021-Present)\n- Built a suite of Tableau dashboards tracking inventory and sales velocity\n  across 60 stores, replacing a manual weekly spreadsheet process.\n- Wrote complex SQL queries against the data warehouse to support ad hoc\n  requests from finance and operations.\n- Analyzed A/B test results for the checkout redesign, recommending the\n  winning variant that li

### Edge case: the loose `resume_john_doe.pdf` (no department subfolder) should get `dept == "general"`

In [5]:
john_doe = next(r for r in records if r["file_name"] == "resume_john_doe.pdf")
assert john_doe["dept"] == "general", john_doe["dept"]
john_doe

{'file_path': 'resumes\\resume_john_doe.pdf',
 'file_name': 'resume_john_doe.pdf',
 'dept': 'general',
 'candidate_name': 'John Doe',
 'skills': ['python',
  'flask',
  'postgresql',
  'docker',
  'git',
  'rest api',
  'django'],
 'total_experience_years': 6.0,
 'education_level': 'Bachelors',
 'sections': {'header': 'John Doe\nSoftware Engineer\njohn.doe@example.com | Remote\n\nGeneralist software engineer with 4 years of experience across backend\nservices and internal tooling, with strong Python experience.',
  'skills': 'Python, Flask, PostgreSQL, Docker, Git, REST APIs',
  'experience': 'Software Engineer, Briarcliff Systems (2022-Present)\n- Built internal tooling in Python (Flask) used by three teams daily.\n- Wrote a Python migration script that moved 2M records between\n  PostgreSQL databases with zero downtime.\nJunior Developer, Oakhollow Software (2020-2022)\n- Maintained a Python/Django billing service.\n- Wrote automated tests in Python, raising coverage from 40% to 85%.

## 4. Build chunks and write chunks.json

In [6]:
chunks = build_chunks(records)

with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Built {len(chunks)} chunks from {len(records)} resumes")
chunks[0]

Built 124 chunks from 31 resumes


{'page_content': 'Raj Patel\nData Analyst\nraj.patel@example.com | Toronto, Canada\n\nData analyst with 5 years turning raw operational data into dashboards and\nrecommendations for product and operations teams.',
 'metadata': {'file_path': 'resumes\\data\\analytics_raj.txt',
  'file_name': 'analytics_raj.txt',
  'dept': 'data',
  'candidate_name': 'Raj Patel',
  'education_level': 'Bachelors',
  'total_experience_years': 7.0,
  'skills': ['sql', 'tableau', 'python', 'ab testing', 'data modeling'],
  'section': 'header'}}

## 5. Generate hybrid dense+sparse embeddings and store in Qdrant

Embeds each chunk's `page_content` twice: a dense vector via OpenRouter's OpenAI-compatible `/embeddings` endpoint (`openai/text-embedding-3-small`), and a sparse BM25 vector via fastembed's `Qdrant/bm25` model. Both are stored as named vectors (`"dense"`/`"sparse"`) on the same point in `config.QDRANT_COLLECTION_NAME`, so a later query-time step can fuse them with Reciprocal Rank Fusion (RRF) — that fusion query itself is deferred to `job_matcher.py`. Payload indexes on `metadata.dept`/`education_level`/`skills` are created for future filtered search.

In [7]:
from resume_rag import (
    build_embedding_model,
    build_sparse_embedding_model,
    embed_chunks,
    ensure_qdrant_collection,
    upsert_chunks,
)

dense_model = build_embedding_model()
sparse_model = build_sparse_embedding_model()
embedded_chunks = embed_chunks(chunks, embedding_model=dense_model, sparse_model=sparse_model)

ensure_qdrant_collection(vector_size=len(embedded_chunks[0]["dense_embedding"]))
upsert_chunks(embedded_chunks)
print(f"Upserted {len(embedded_chunks)} chunks into '{config.QDRANT_COLLECTION_NAME}'")

Upserted 124 chunks into 'resume_chunks'


In [8]:
count = config.client.count(config.QDRANT_COLLECTION_NAME).count
print(f"Collection '{config.QDRANT_COLLECTION_NAME}' has {count} points")
assert count == len(chunks), (count, len(chunks))

sample_point = config.client.retrieve(
    config.QDRANT_COLLECTION_NAME,
    ids=[config.client.scroll(config.QDRANT_COLLECTION_NAME, limit=1)[0][0].id],
    with_vectors=True,
)[0]

assert sample_point.vector is not None and isinstance(sample_point.vector, dict)
print("dense vector length:", len(sample_point.vector["dense"]))
print("sparse vector nnz:", len(sample_point.vector["sparse"].indices))
sample_point.payload

Collection 'resume_chunks' has 124 points
dense vector length: 1536
sparse vector nnz: 9


{'page_content': 'Salesforce, channel partnerships, contract negotiation, co-selling,\npartner enablement',
 'metadata': {'file_path': 'resumes\\sales\\partnerships_omar.pdf',
  'file_name': 'partnerships_omar.pdf',
  'dept': 'sales',
  'candidate_name': 'Omar Haddad',
  'education_level': 'Bachelors',
  'total_experience_years': 8.0,
  'skills': ['salesforce',
   'channel partnerships',
   'contract negotiation',
   'co-selling',
   'partner enablement'],
  'section': 'skills'}}

## 6. Match a job description against indexed resumes

`job_matcher.match_job` embeds the JD's descriptive prose (hybrid dense+sparse), runs a Qdrant `query_points` hybrid search fusing both with Reciprocal Rank Fusion (RRF), then hard-filters out any candidate failing a must-have requirement (checked algorithmically against `total_experience_years` + normalized skills, not an LLM call) and ranks survivors 0-100.

In [9]:
from fs_tools import read_file
from job_matcher import match_job

sample_job_path = "jobs/senior_backend_engineer.txt"
jd_text = read_file.invoke({"filepath": sample_job_path})["content"]

matches = match_job(jd_text, top_k=config.MATCH_TOP_K)
for m in matches:
    print(f"{m.match_score:5.1f}  {m.candidate_name:20s}  {m.resume_path}")
    print(f"       skills: {', '.join(m.matched_skills)}")
    print(f"       {m.reasoning}")
    print()

INFO: read_file(filepath='jobs/senior_backend_engineer.txt')


100.0  Alice Chen            resumes\engineering\backend_alice.txt
       skills: django, fastapi, postgresql, python
       Matched via hybrid dense+sparse similarity (RRF score 0.8333); satisfied all 4 must-have requirement(s), including django, fastapi, postgresql, python.

100.0  Wei Zhang             resumes\engineering\fullstack_wei.docx
       skills: django, fastapi, postgresql, python
       Matched via hybrid dense+sparse similarity (RRF score 0.8333); satisfied all 4 must-have requirement(s), including django, fastapi, postgresql, python.

 24.2  Neha Verma            resumes\data\data_engineer_neha.txt
       skills: python, sql
       Matched via hybrid dense+sparse similarity (RRF score 0.2576); satisfied all 4 must-have requirement(s), including python, sql.

 10.6  John Doe              resumes\resume_john_doe.pdf
       skills: django, postgresql, python
       Matched via hybrid dense+sparse similarity (RRF score 0.1546); satisfied all 4 must-have requirement(s), incl

### Trying more job postings

Two harder cases: a "5+ years" vs. "3+ years" quantifier difference and a job whose must-haves are mostly unquantified (portfolio/mentoring-style) rather than tech-keyword bullets.

In [10]:
for extra_job_path in ["jobs/machine_learning_engineer.txt", "jobs/senior_product_designer.txt"]:
    extra_jd_text = read_file.invoke({"filepath": extra_job_path})["content"]
    extra_matches = match_job(extra_jd_text, top_k=config.MATCH_TOP_K)
    print(f"=== {extra_job_path} ({len(extra_matches)} matches) ===")
    for m in extra_matches:
        print(f"{m.match_score:5.1f}  {m.candidate_name:20s}  {m.resume_path}")
    print()

INFO: read_file(filepath='jobs/machine_learning_engineer.txt')


=== jobs/machine_learning_engineer.txt (2 matches) ===
100.0  Sara Kim              resumes\data\ml_sara.docx
  0.0  Farah Aziz            resumes\engineering\mlops_farah.pdf

INFO: read_file(filepath='jobs/senior_product_designer.txt')


=== jobs/senior_product_designer.txt (1 matches) ===
100.0  Priyanka Rao          resumes\design\design_lead_priyanka.pdf



## 7. Performance metrics: retrieval accuracy + latency

`JOB_GROUND_TRUTH` below is a hand-labeled mapping of job path -> resume paths a human reviewer expects to see matched, derived by reading each of the 6 job postings' must-haves against all 31 resumes' extracted skills/experience.

In [11]:
from metrics import measure_latency, measure_retrieval_accuracy, summarize_latencies

JOB_GROUND_TRUTH = {
    "jobs/senior_backend_engineer.txt": [
        r"resumes\engineering\backend_alice.txt",
        r"resumes\engineering\fullstack_wei.docx",
        r"resumes\resume_john_doe.pdf",
    ],
    "jobs/data_engineer.txt": [
        r"resumes\data\data_engineer_neha.txt",
    ],
    "jobs/machine_learning_engineer.txt": [
        r"resumes\data\ml_sara.docx",
        r"resumes\engineering\mlops_farah.pdf",
    ],
    "jobs/product_marketing_manager.txt": [
        r"resumes\marketing\product_marketing_elena.docx",
    ],
    "jobs/senior_product_designer.txt": [
        r"resumes\design\design_lead_priyanka.pdf",
    ],
    "jobs/enterprise_account_executive.txt": [
        r"resumes\sales\account_exec_marcus.pdf",
        r"resumes\sales\enterprise_dan.txt",
    ],
}

accuracy_results = measure_retrieval_accuracy(JOB_GROUND_TRUTH, top_k=config.MATCH_TOP_K)
for r in accuracy_results:
    print(f"{r.job_path:45s}  precision@k={r.precision_at_k:.2f}  recall@k={r.recall_at_k:.2f}  retrieved={len(r.retrieved_resume_paths)}")
accuracy_results

INFO: read_file(filepath='jobs/senior_backend_engineer.txt')


INFO: read_file(filepath='jobs/data_engineer.txt')


INFO: read_file(filepath='jobs/machine_learning_engineer.txt')


INFO: read_file(filepath='jobs/product_marketing_manager.txt')


INFO: read_file(filepath='jobs/senior_product_designer.txt')


INFO: read_file(filepath='jobs/enterprise_account_executive.txt')


jobs/senior_backend_engineer.txt               precision@k=0.43  recall@k=1.00  retrieved=7
jobs/data_engineer.txt                         precision@k=1.00  recall@k=1.00  retrieved=1
jobs/machine_learning_engineer.txt             precision@k=1.00  recall@k=1.00  retrieved=2
jobs/product_marketing_manager.txt             precision@k=1.00  recall@k=1.00  retrieved=1
jobs/senior_product_designer.txt               precision@k=1.00  recall@k=1.00  retrieved=1
jobs/enterprise_account_executive.txt          precision@k=0.00  recall@k=0.00  retrieved=0


[RetrievalAccuracyResult(job_path='jobs/senior_backend_engineer.txt', expected_resume_paths=['resumes\\engineering\\backend_alice.txt', 'resumes\\engineering\\fullstack_wei.docx', 'resumes\\resume_john_doe.pdf'], retrieved_resume_paths=['resumes\\engineering\\fullstack_wei.docx', 'resumes\\engineering\\backend_alice.txt', 'resumes\\data\\data_engineer_neha.txt', 'resumes\\resume_john_doe.pdf', 'resumes\\data\\analytics_raj.txt', 'resumes\\data\\product_analyst_owen.pdf', 'resumes\\data\\data_scientist_hana.docx'], precision_at_k=0.42857142857142855, recall_at_k=1.0),
 RetrievalAccuracyResult(job_path='jobs/data_engineer.txt', expected_resume_paths=['resumes\\data\\data_engineer_neha.txt'], retrieved_resume_paths=['resumes\\data\\data_engineer_neha.txt'], precision_at_k=1.0, recall_at_k=1.0),
 RetrievalAccuracyResult(job_path='jobs/machine_learning_engineer.txt', expected_resume_paths=['resumes\\data\\ml_sara.docx', 'resumes\\engineering\\mlops_farah.pdf'], retrieved_resume_paths=['resu

In [12]:
latency, matches = measure_latency("match_job", match_job, jd_text, top_k=config.MATCH_TOP_K)
summarize_latencies([latency])

{'match_job': {'count': 1,
  'min': 0.6077181000000564,
  'mean': 0.6077181000000564,
  'max': 0.6077181000000564,
  'p95': 0.6077181000000564}}